In [ ]:
#!/usr/bin/env python3
# updated_pipeline_wilcoxon.py
import os
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.spatial import distance
from scipy import stats
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
from collections import Counter
import random
import copy

# -------------------------
# Feature extraction and classifier classes (kept and lightly refactored)
# -------------------------
class ExanthemClassifier:
    def __init__(self):
        self.models = [
            ('RandomForest', RandomForestClassifier(n_estimators=100, random_state=42)),
            ('SVM', SVC(probability=True, kernel='rbf', class_weight='balanced', random_state=42)),
            ('KNN', KNeighborsClassifier(n_neighbors=5))
        ]
        self.scaler = StandardScaler()
        self.feature_names = ['lesion_count', 'avg_area', 'std_area', 'avg_circularity',
                              'sparsity_score', 'confluence_ratio', 'avg_hue', 'avg_saturation']

    def apply_gray_world_white_balance(self, img):
        b, g, r = cv2.split(img.astype(np.float32))
        avg_b, avg_g, avg_r = np.mean(b), np.mean(g), np.mean(r)
        avg_all = (avg_b + avg_g + avg_r) / 3.0
        scale_b = avg_all / avg_b if avg_b > 0 else 1.0
        scale_g = avg_all / avg_g if avg_g > 0 else 1.0
        scale_r = avg_all / avg_r if avg_r > 0 else 1.0
        b = np.clip(b * scale_b, 0, 255)
        g = np.clip(g * scale_g, 0, 255)
        r = np.clip(r * scale_r, 0, 255)
        return cv2.merge((b, g, r)).astype(np.uint8)

    def extract_features(self, image_path):
        img = cv2.imread(image_path)
        if img is None:
            return None
        img = cv2.resize(img, (512, 512))
        smoothed = cv2.bilateralFilter(img, d=9, sigmaColor=75, sigmaSpace=75)
        gray = cv2.cvtColor(smoothed, cv2.COLOR_BGR2GRAY)
        clahe = cv2.createCLAHE(clipLimit=1.5, tileGridSize=(8, 8))
        equalized = clahe.apply(gray)
        thresh = cv2.adaptiveThreshold(equalized, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                                       cv2.THRESH_BINARY_INV, 51, 2)
        kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7, 7))
        clean_thresh = cv2.morphologyEx(thresh, cv2.MORPH_OPEN, kernel, iterations=1)
        contours, _ = cv2.findContours(clean_thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

        centroids, areas, circularities = [], [], []
        valid_contours = []
        for cnt in contours:
            area = cv2.contourArea(cnt)
            if 50 < area < (512 * 512 * 0.1):
                perimeter = cv2.arcLength(cnt, True)
                circularity = 0 if perimeter == 0 else 4 * np.pi * (area / (perimeter * perimeter))
                M = cv2.moments(cnt)
                if M["m00"] != 0:
                    centroids.append((int(M["m10"] / M["m00"]), int(M["m01"] / M["m00"])))
                    areas.append(area)
                    circularities.append(circularity)
                    valid_contours.append(cnt)

        wb_img = self.apply_gray_world_white_balance(img)
        hsv_img = cv2.cvtColor(wb_img, cv2.COLOR_BGR2HSV)
        valid_mask = np.zeros((512, 512), dtype=np.uint8)
        if valid_contours:
            cv2.drawContours(valid_mask, valid_contours, -1, 255, thickness=cv2.FILLED)
            mean_color = cv2.mean(hsv_img, mask=valid_mask)
            avg_hue = mean_color[0]
            avg_saturation = mean_color[1]
        else:
            avg_hue = 0
            avg_saturation = 0

        std_area = np.std(areas) if len(areas) > 1 else 0
        avg_circularity = np.mean(circularities) if circularities else 0

        if len(centroids) > 1:
            dist_matrix = distance.cdist(centroids, centroids, 'euclidean')
            np.fill_diagonal(dist_matrix, np.inf)
            nn_distances = np.min(dist_matrix, axis=1)
            sparsity_score = np.mean(nn_distances)
        else:
            sparsity_score = 0

        confluence_ratio = sum(areas) / (512 * 512)
        return [len(centroids), np.mean(areas) if areas else 0, std_area,
                avg_circularity, sparsity_score, confluence_ratio, avg_hue, avg_saturation]

    def load_dataset(self, root_dir):
        data, labels, filenames = [], [], []
        if not os.path.isdir(root_dir):
            return np.array([]), np.array([]), []
        for label_dir in sorted(os.listdir(root_dir)):
            dir_path = os.path.join(root_dir, label_dir)
            if not os.path.isdir(dir_path):
                continue
            for img_file in sorted(os.listdir(dir_path)):
                img_path = os.path.join(dir_path, img_file)
                f_vector = self.extract_features(img_path)
                if f_vector:
                    data.append(f_vector)
                    labels.append(label_dir)
                    filenames.append(img_file)
        return np.array(data), np.array(labels), filenames

# -------------------------
# Utility functions for balancing and statistics
# -------------------------
def balance_dataset_undersample(X, y, filenames):
    """Undersample majority classes to match the smallest class count."""
    counts = Counter(y)
    if len(counts) <= 1:
        return X, y, filenames
    min_count = min(counts.values())
    indices_per_class = {cls: np.where(y == cls)[0].tolist() for cls in counts.keys()}
    selected_indices = []
    for cls, idxs in indices_per_class.items():
        if len(idxs) <= min_count:
            selected = idxs
        else:
            selected = random.sample(idxs, min_count)
        selected_indices.extend(selected)
    selected_indices = sorted(selected_indices)
    return X[selected_indices], y[selected_indices], [filenames[i] for i in selected_indices]

def compute_ci(scores, confidence=0.95):
    """Compute mean, std, and t-based confidence interval for a 1D array of scores."""
    a = np.array(scores)
    n = len(a)
    mean = a.mean()
    std = a.std(ddof=1) if n > 1 else 0.0
    if n > 1:
        se = stats.sem(a)
        h = se * stats.t.ppf((1 + confidence) / 2., n - 1)
        return mean, std, (mean - h, mean + h)
    else:
        return mean, std, (mean, mean)

# -------------------------
# Explainable diagnostics (kept intact)
# -------------------------
class ExplainableDiagnostics:
    def __init__(self, classifier_obj):
        self.c = classifier_obj

    def analyze_failures(self, model, X_test_norm, y_test, filenames, model_name):
        predictions = model.predict(X_test_norm)
        errors = []
        for i in range(len(y_test)):
            if predictions[i] != y_test[i]:
                errors.append({
                    'filename': filenames[i],
                    'actual': y_test[i],
                    'predicted': predictions[i]
                })
        df_err = pd.DataFrame(errors)
        print(f"\n--- Failure Analysis: {model_name} ---")
        if not df_err.empty:
            print(f"Total Errors: {len(df_err)}")
            print(df_err.head(10))
        return df_err

    def plot_performance(self, model, X_test, y_test, model_name):
        y_pred = model.predict(X_test)
        cm = confusion_matrix(y_test, y_pred)
        plt.figure(figsize=(12, 5))
        plt.subplot(1, 2, 1)
        sns.heatmap(cm, annot=True, fmt='d', cmap='Reds',
                    xticklabels=np.unique(y_test), yticklabels=np.unique(y_test))
        plt.title(f'Confusion Matrix: {model_name}')
        plt.ylabel('Actual Label')
        plt.xlabel('Predicted Label')
        if hasattr(model, 'feature_importances_'):
            plt.subplot(1, 2, 2)
            importances = model.feature_importances_
            indices = np.argsort(importances)
            plt.barh(range(len(indices)), importances[indices], align='center', color='steelblue')
            plt.yticks(range(len(indices)), [self.c.feature_names[i] for i in indices])
            plt.title('Feature Importance')
        plt.tight_layout()
        plt.show()

    def visualize_explainable_errors(self, df_errors, root_dir, num_samples=5):
        if df_errors.empty:
            return
        samples = df_errors.head(num_samples)
        for _, row in samples.iterrows():
            img_path = os.path.join(root_dir, row['actual'], row['filename'])
            img = cv2.imread(img_path)
            if img is None:
                continue
            img = cv2.resize(img, (512, 512))
            wb_img = self.c.apply_gray_world_white_balance(img)
            smoothed = cv2.bilateralFilter(img, d=9, sigmaColor=75, sigmaSpace=75)
            gray = cv2.cvtColor(smoothed, cv2.COLOR_BGR2GRAY)
            clahe = cv2.createCLAHE(clipLimit=1.5, tileGridSize=(8, 8))
            equalized = clahe.apply(gray)
            thresh = cv2.adaptiveThreshold(equalized, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                                           cv2.THRESH_BINARY_INV, 51, 2)
            kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7, 7))
            clean_thresh = cv2.morphologyEx(thresh, cv2.MORPH_OPEN, kernel, iterations=1)
            contours, _ = cv2.findContours(clean_thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
            centroid_map = np.zeros((512, 512), dtype=np.uint8)
            for cnt in contours:
                area = cv2.contourArea(cnt)
                if 50 < area < (512 * 512 * 0.1):
                    M = cv2.moments(cnt)
                    if M["m00"] != 0:
                        cX, cY = int(M["m10"] / M["m00"]), int(M["m01"] / M["m00"])
                        cv2.circle(centroid_map, (cX, cY), 5, 255, -1)
            fig, axes = plt.subplots(1, 4, figsize=(22, 6))
            fig.suptitle(f"XAI FAILURE ANALYSIS | File: {row['filename']} | Actual: {row['actual']} | Predicted: {row['predicted']}",
                         fontsize=14, fontweight='bold')
            axes[0].imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB)); axes[0].set_title("1. Original Image")
            axes[1].imshow(cv2.cvtColor(wb_img, cv2.COLOR_BGR2RGB)); axes[1].set_title("2. WB Color Corrected")
            axes[2].imshow(clean_thresh, cmap='gray'); axes[2].set_title("3. CLAHE + Cleaned Mask")
            axes[3].imshow(centroid_map, cmap='magma'); axes[3].set_title("4. Spatial Centroid Map")
            for ax in axes: ax.axis('off')
            plt.tight_layout()
            plt.show()

# -------------------------
# Main pipeline execution with repeated CV and Wilcoxon stability tests
# -------------------------
def main(train_dir='./train', test_dir='./test', n_splits=5, repeat_cv_runs=10,
         repeat_test_runs=1, random_seed=42, alpha=0.05):
    """
    - n_splits: number of folds for CV (use 5 as requested)
    - repeat_cv_runs: how many times to repeat the full k-fold CV with different shuffles (default 10)
    - repeat_test_runs: how many times to retrain/evaluate on test set for test stability (default 1)
    - alpha: significance threshold for Wilcoxon tests
    """
    random.seed(random_seed)
    np.random.seed(random_seed)

    pipeline = ExanthemClassifier()
    diagnostics = ExplainableDiagnostics(pipeline)

    print("--- Phase 1: Training & Repeated Cross-Validation (stability analysis) ---")
    X_train_raw, y_train, train_filenames = pipeline.load_dataset(train_dir)
    if len(X_train_raw) == 0:
        print("Error: No training data found. Check directory structure.")
        return

    # Balance training set by undersampling to smallest class
    X_train_bal, y_train_bal, train_filenames_bal = balance_dataset_undersample(X_train_raw, y_train, train_filenames)
    print(f"Original training class distribution: {Counter(y_train)}")
    print(f"Balanced training class distribution: {Counter(y_train_bal)}")

    # Fit scaler on balanced training set only
    X_train = pipeline.scaler.fit_transform(X_train_bal)
    print(f"Dataset Loaded: {len(X_train)} training samples with {X_train.shape[1]} features.")

    # For each model: perform repeated stratified k-fold CV, collect per-fold accuracies per run
    model_cv_runs = {name: [] for name, _ in pipeline.models}  # each entry: list of arrays (length n_splits)
    for run in range(repeat_cv_runs):
        run_seed = random_seed + run
        skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=run_seed)
        print(f"\nCV repeat {run + 1}/{repeat_cv_runs} (seed={run_seed})")
        for name, base_model in pipeline.models:
            model = copy.deepcopy(base_model)
            # cross-validation manual loop to collect fold scores (so we can keep fold order)
            fold_scores = []
            for train_idx, val_idx in skf.split(X_train, y_train_bal):
                X_tr, X_val = X_train[train_idx], X_train[val_idx]
                y_tr, y_val = y_train_bal[train_idx], y_train_bal[val_idx]
                # fit model on fold training
                model.fit(X_tr, y_tr)
                y_pred = model.predict(X_val)
                acc = accuracy_score(y_val, y_pred)
                fold_scores.append(acc)
            model_cv_runs[name].append(np.array(fold_scores))
            mean_run = np.mean(fold_scores)
            std_run = np.std(fold_scores, ddof=1) if len(fold_scores) > 1 else 0.0
            print(f"{name} run {run+1} mean acc: {mean_run:.4f} (+/- {std_run:.4f})")

    # Aggregate CV results per model across repeats
    print("\n--- Aggregated CV statistics per model (all repeats combined) ---")
    for name in model_cv_runs:
        # flatten all fold scores across repeats
        all_fold_scores = np.concatenate(model_cv_runs[name])
        mean_all, std_all, (ci_low, ci_high) = compute_ci(all_fold_scores, confidence=0.95)
        print(f"\n{name} aggregated CV accuracy (all folds across repeats):")
        print(f"  mean = {mean_all:.4f}, std = {std_all:.4f}, 95% CI = [{ci_low:.4f}, {ci_high:.4f}]")
        # Also show per-run means for transparency
        per_run_means = [arr.mean() for arr in model_cv_runs[name]]
        print(f"  per-run means: {[f'{m:.4f}' for m in per_run_means]}")

    # Wilcoxon signed-rank test for stability per model:
    # Compare run 1 fold scores vs each subsequent run's fold scores (paired by fold index)
    print("\n--- Wilcoxon signed-rank tests (stability across CV repeats) ---")
    for name in model_cv_runs:
        runs = model_cv_runs[name]
        if len(runs) < 2:
            print(f"{name}: only one CV run available; cannot perform Wilcoxon test.")
            continue
        base = runs[0]
        print(f"\n{name}: comparing run 1 to other runs (alpha={alpha})")
        for idx in range(1, len(runs)):
            other = runs[idx]
            # Wilcoxon requires paired samples of same length; both are length n_splits
            try:
                stat, p = stats.wilcoxon(base, other, zero_method='wilcox', alternative='two-sided', mode='approx')
            except ValueError:
                # fallback if all differences are zero or other numerical issues
                stat, p = np.nan, 1.0
            stable = 'stable' if (not np.isnan(p) and p >= alpha) else 'unstable'
            print(f"  run1 vs run{idx+1}: W={stat}, p={p:.4f} -> {stable}")

    # -------------------------
    # Phase 2: Final training on full balanced training set and test evaluation
    # -------------------------
    print("\n--- Phase 2: Final training on full balanced training set & Unseen Test Evaluation ---")
    # Train final models on full balanced training set
    trained_models = {}
    for name, base_model in pipeline.models:
        model = copy.deepcopy(base_model)
        model.fit(X_train, y_train_bal)
        trained_models[name] = model

    # Load and balance test set (undersample)
    X_test_raw, y_test, test_filenames = pipeline.load_dataset(test_dir)
    if len(X_test_raw) == 0:
        print("Error: No testing data found. Check directory structure.")
        return
    X_test_bal, y_test_bal, test_filenames_bal = balance_dataset_undersample(X_test_raw, y_test, test_filenames)
    print(f"Original test class distribution: {Counter(y_test)}")
    print(f"Balanced test class distribution: {Counter(y_test_bal)}")
    X_test = pipeline.scaler.transform(X_test_bal)

    # Evaluate final models once on test set (or repeat if requested)
    test_run_results = {name: [] for name in trained_models.keys()}
    test_run_predictions = {name: None for name in trained_models.keys()}

    for run in range(repeat_test_runs):
        run_seed = random_seed + run
        print(f"\nTest run {run+1}/{repeat_test_runs} (seed={run_seed})")
        for name, base_model in pipeline.models:
            model = copy.deepcopy(base_model)
            if hasattr(model, 'random_state'):
                try:
                    model.random_state = run_seed
                except Exception:
                    pass
            # retrain on full balanced training set for each test run
            model.fit(X_train, y_train_bal)
            y_pred = model.predict(X_test)
            acc = accuracy_score(y_test_bal, y_pred)
            test_run_results[name].append(acc)
            test_run_predictions[name] = (y_pred, model)
            if run == 0:
                print(f"\n{name} classification report (test run 1):")
                print(classification_report(y_test_bal, y_pred))

    # Summarize test results
    print("\n--- Test set summary ---")
    for name in test_run_results:
        scores = test_run_results[name]
        mean_acc, std_acc, (ci_low, ci_high) = compute_ci(scores, confidence=0.95)
        if len(scores) == 1:
            print(f"{name} Test Accuracy: {scores[0]:.4f} (single run)")
        else:
            print(f"{name} Test Accuracy (mean over runs): {mean_acc:.4f} (+/- {std_acc:.4f})")
            print(f"{name} Test 95% CI: [{ci_low:.4f}, {ci_high:.4f}]")

    # Final XAI diagnostics using last trained models/predictions
    for name in test_run_predictions:
        y_pred, model_for_xai = test_run_predictions[name]
        print(f"\nFinal diagnostics for {name}:")
        print(classification_report(y_test_bal, y_pred))
        df_errors = diagnostics.analyze_failures(model_for_xai, X_test, y_test_bal, test_filenames_bal, name)
        diagnostics.plot_performance(model_for_xai, X_test, y_test_bal, name)
        if name == 'RandomForest':
            print(f"Generating XAI Visualizations for {name} misclassifications...")
            diagnostics.visualize_explainable_errors(df_errors, test_dir, num_samples=5)

    print("\nPipeline complete. Repeated CV performed on training set with Wilcoxon stability tests per model; test set evaluated separately as final generalization check.")

if __name__ == "__main__":
    # Defaults:
    # - n_splits=5 (as requested)
    # - repeat_cv_runs=10 (recommended to get multiple runs for Wilcoxon comparisons)
    # - repeat_test_runs=1 (single final test evaluation)
    main(train_dir='./train', test_dir='./test', n_splits=5, repeat_cv_runs=10, repeat_test_runs=1, random_seed=42, alpha=0.05)
